# From a spectrum to a structure: which acid gave this infrared spectrum?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igbaskin/cmf-qsar/blob/main/docs/tutorials/30_spectrum_to_structure.ipynb)

An infrared spectrum is often recorded to answer one question: which of the possible
structures is this compound? The usual tool is a **library search**. The spectrum of the unknown
(the query) is compared with the spectra of known compounds (the library), and the candidates are
ranked by how alike the spectra are.

When the library is computed, its spectra and the query come from different sources. Their lines
sit at slightly different places, and the search has to tolerate that. This notebook measures how
well the spectrum kernel of `cmfqsar.spectrum` does it. The kernel puts a Gaussian of a chosen
width on every line and compares the two smooth curves. It is set against the searches a
spectroscopist would use first: a comparison of binned spectra and a comparison of peak tables.

The set is the 189 substituted benzoic acids of plan items SP2 and SP3. The library holds one
spectrum per acid, computed with ORCA at the r2SCAN-3c level. The queries are the spectra of the
same acids from two other methods, used as stand-ins for a measurement:

- **B97-3c**, a second density functional, with its own optimised geometry;
- **GFN2-xTB**, a much faster and rougher semi-empirical method.

The notebook asks three questions.

1. How often is the right acid ranked first, by the kernel and by the cheap searches?
2. Why does the kernel do better, and what limits it on a real measured spectrum?
3. Without any target, can a pairing of structure and spectrum show which part of the structure
   moves which part of the spectrum? And what does the pairing add over each of the two alone?

Running time: about one minute.

In [1]:
# Run this cell first. On Google Colab it fetches the repository and installs the package;
# on a local checkout it only moves to the repository root, so that the example paths work.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("cmf-qsar"):
        subprocess.run(["git", "clone", "-q", "https://github.com/igbaskin/cmf-qsar.git"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./cmf-qsar", "py3Dmol"], check=True)
    os.chdir("cmf-qsar")
else:
    while not os.path.exists("pyproject.toml") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")
print("working in", os.getcwd())

working in C:\Users\baski\work\claude\cmf


## The data

Each spectrum is a list of lines: a position in cm^-1 and an intensity. The meta and the para acid
of one substituent have the same molecular formula, so the composition alone cannot tell them
apart. The cell counts how often the formula alone would put the right acid first. It also shows
how far the lines of each query level are from the library: the carbonyl band and the highest
line, which is the O-H stretch in most acids.

In [2]:
import json
import time

import numpy as np
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
sys.path.insert(0, "benchmarks")
import run_spectra_bridge as BR                 # the benchmark script of plan item SP3

start = time.time()
names, table = BR.read_names()
library = BR.IR.read_spectra(BR.LIBRARY, names)
queries = {level: BR.IR.read_spectra(BR.LEVELS[level]["path"], names)
           for level in ("b97-3c", "xtb")}
label = {level: BR.LEVELS[level]["label"] for level in queries}
formula = BR.formula_scores(table)
print("%d acids; the library spectra have %d to %d lines"
      % (len(names), min(len(s) for s in library), max(len(s) for s in library)))
print("the composition alone puts the right acid first in %.1f %% of the queries "
      "(the largest group with one formula has %d acids)"
      % (100 * formula["top1"], formula["largest_group"]))


def carbonyl(s):
    part = s.within(*BR.CARBONYL)
    return float(part.x[int(np.argmax(part.y))])


for level, qs in queries.items():
    co = np.array([carbonyl(q) - carbonyl(l) for q, l in zip(qs, library)])
    top = np.array([q.x[-1] - l.x[-1] for q, l in zip(qs, library)])
    print("%-9s carbonyl band %+6.1f cm^-1 from the library (sd %4.1f), highest line %+7.1f (sd %4.1f)"
          % (label[level], co.mean(), co.std(), top.mean(), top.std()))

189 acids; the library spectra have 39 to 105 lines
the composition alone puts the right acid first in 45.0 % of the queries (the largest group with one formula has 4 acids)
B97-3c    carbonyl band   -4.8 cm^-1 from the library (sd  3.0), highest line   -26.6 (sd  1.5)
GFN2-xTB  carbonyl band  +29.3 cm^-1 from the library (sd  4.4), highest line  -182.0 (sd 18.7)


## Four ways to rank the candidates

For every query, all 189 library spectra are ranked, and the rank of the right acid is recorded.
Four searches are compared.

- **The position of the carbonyl band alone.** The bands of the query level are put on the scale
  of the library by a straight line, and the candidates are ranked by how close their band is.
- **Binned spectra.** Every spectrum becomes a histogram: the intensities of its lines are summed
  in bins of one width. The candidates are ranked by the cosine of the two histograms.
- **A peak-table search.** Every spectrum becomes the list of its stronger lines. The candidates
  are ranked by the share of the peaks of both lists that have a partner in the other list within
  a tolerance.
- **The spectrum kernel.** Every line becomes a Gaussian of a chosen width. The candidates are
  ranked by the cosine of the two smooth curves. Both levels are **centred** first: the mean
  spectrum of each level is subtracted. All the acids share the bands of benzoic acid, and the
  centring leaves what each acid has beyond those bands.

Each search has settings: the bin width, the share of the strongest line and the tolerance, the
width of the Gaussian. A setting must not be chosen on the queries it is scored on. So the
substituents are split in half. The setting with the best ranks on the acids of one half (the
calibration acids) is used on the queries of the other half (the test acids). The split is
repeated ten times, and every search uses the same splits.

In [3]:
twins = BR.twin_rows(table)
formulas = np.array(table["formula"])
same = formulas[:, None] == formulas[None, :]
splits = [BR.split_substituents(table, r) for r in range(BR.REPEATS)]
chosen, rows = {}, []
for level, qs in queries.items():
    families = BR.reference_settings(library, qs)
    chosen[level] = {fam: BR.choose_on_calibration(s, splits, twins, same)
                     for fam, s in families.items()}
    band = [BR.band_scores(library, qs, cal, test, twins) for cal, test in splits]
    for fam, runs in [("the carbonyl band alone", band), ("binned spectra", chosen[level]["binned"]),
                      ("a peak-table search", chosen[level]["peaks"]),
                      ("the spectrum kernel", chosen[level]["kernel"])]:
        first = 100 * np.array([r["top1"] for r in runs])
        top3 = 100 * np.mean([r["top3"] for r in runs])
        isomer = 100 * np.mean([r["isomer"] for r in runs])
        rows.append((label[level], fam, first.mean(), first.std(), top3, isomer))
print("%-9s %-24s %14s %15s %22s" % ("query", "search", "first, % (sd)", "first three, %",
                                    "right isomer above, %"))
for r in rows:
    print("%-9s %-24s %8.1f (%3.1f) %15.1f %22.1f" % r)
print("\n(%.0f s)" % (time.time() - start))

query     search                    first, % (sd)  first three, %  right isomer above, %
B97-3c    the carbonyl band alone       6.4 (3.3)            17.2                   83.1
B97-3c    binned spectra               77.0 (4.6)            90.6                   91.3
B97-3c    a peak-table search          86.8 (2.9)            98.2                   99.1
B97-3c    the spectrum kernel          99.6 (0.5)            99.7                  100.0
GFN2-xTB  the carbonyl band alone       2.7 (1.4)             7.8                   70.5
GFN2-xTB  binned spectra               32.9 (2.6)            60.3                   71.0
GFN2-xTB  a peak-table search           8.2 (1.9)            22.1                   63.7
GFN2-xTB  the spectrum kernel          42.8 (2.6)            67.7                   74.0

(39 s)


In [4]:
for level in queries:
    print(label[level])
    for fam in ("binned", "peaks"):
        d = BR.paired_difference(chosen[level]["kernel"], chosen[level][fam], table)
        print("  the kernel minus %-19s %+5.1f points [%+5.1f, %+5.1f]; kernel ahead in %d of 10 splits"
              % ("binned spectra:" if fam == "binned" else "the peak tables:", 100 * d["mean"],
                 100 * d["low"], 100 * d["high"], d["ahead"]))
    for fam in ("binned", "peaks", "kernel"):
        settings = [r["setting"] for r in chosen[level][fam]]
        common = max(set(settings), key=settings.count)
        print("  %-8s chosen most often: %s (%d of 10 splits)"
              % (fam, common, settings.count(common)))

B97-3c
  the kernel minus binned spectra:     +22.5 points [+17.6, +27.8]; kernel ahead in 10 of 10 splits


  the kernel minus the peak tables:    +12.7 points [ +7.3, +18.6]; kernel ahead in 10 of 10 splits
  binned   chosen most often: bins of 100 cm^-1, power 0.5, centred (5 of 10 splits)
  peaks    chosen most often: lines of 5 % of the strongest or more, within 20 cm^-1 (10 of 10 splits)
  kernel   chosen most often: 20 cm^-1, power 0.5 (7 of 10 splits)
GFN2-xTB
  the kernel minus binned spectra:      +9.9 points [ +3.4, +16.7]; kernel ahead in 10 of 10 splits


  the kernel minus the peak tables:    +34.6 points [+27.1, +42.4]; kernel ahead in 10 of 10 splits
  binned   chosen most often: bins of 200 cm^-1, power 0.5, centred (10 of 10 splits)
  peaks    chosen most often: lines of 10 % of the strongest or more, within 60 cm^-1 (8 of 10 splits)
  kernel   chosen most often: 50 cm^-1, power 0.5 (5 of 10 splits)


**With a B97-3c query the kernel puts the right acid first in 99.6 % of the test queries.** The
better of the two everyday searches, the peak table, reaches 86.8 %; the binned spectra reach
77.0 %. The kernel is ahead of both in all ten splits: by +12.7 percentage points [+7.3, +18.6]
over the peak tables and by +22.5 [+17.6, +27.8] over the binned spectra. The interval comes from
resampling whole substituents. The carbonyl band alone finds the acid in 6.4 % of the cases: one
number cannot tell 189 acids apart.

**With a GFN2-xTB query every search does much worse, and the kernel still leads.** It finds the
acid in 42.8 %, against 32.9 % for the binned spectra (+9.9 [+3.4, +16.7]) and 8.2 % for the peak
tables. That is below the 45.0 % of the composition alone, which knows the formula exactly. On
the question the composition cannot answer, which of two isomers, the lead is small: the kernel
puts the right isomer above its meta or para twin in 74.0 %, the binned spectra in 71.0 %. All
three searches choose wide settings here (a Gaussian of 50 cm^-1, bins of 200 cm^-1, a tolerance
of 60 cm^-1): the lines of GFN2-xTB agree with the library only region by region.

**A note on fairness.** The binned search was first offered bins up to 100 cm^-1, and on the
GFN2-xTB queries it chose the widest. A reference that chooses the edge of its grid may be held
back by the grid. With bins of 200 and 400 cm^-1 added it rose from 23.5 % to 32.9 %, and it now
chooses 200, inside the grid. The kernel's lead fell from +19.3 to +9.9 points. The same check
on the tolerance of the peak tables changed nothing.

## Why the smooth curve wins: the edge of a bin

Take two spectra with one line each, 5 cm^-1 apart. With bins 20 cm^-1 wide, the two lines fall
in the same bin or in neighbouring bins, depending on where the bin edge happens to lie. In the
first case the histograms agree completely; in the second they have nothing in common. A
peak table has the same problem at the edge of its tolerance.

The kernel has no edge. Two Gaussians of width sigma whose centres are d apart overlap by
exp(-d^2 / 4 sigma^2), which falls smoothly as d grows. The cell compares the two for a pair of
lines, averaged over every position of the bin edge.

In [5]:
step = sigma = 20.0
print("%14s %28s %22s" % ("distance, cm^-1", "binned: mean match (range)", "kernel: overlap"))
for d in (0.0, 2.0, 5.0, 10.0, 20.0, 40.0):
    same_bin = max(0.0, 1.0 - d / step)          # share of edge positions that keep both lines in one bin
    print("%14.0f %16.2f (0 to %d) %22.2f" % (d, same_bin, 1 if d < step else 0,
                                              np.exp(-d * d / (4 * sigma * sigma))))
gaps = np.concatenate([np.abs(q.x[:, None] - l.x[None, :]).min(axis=1)
                       for q, l in zip(queries["b97-3c"], library)])
print("\nB97-3c query lines: median distance to the nearest library line %.1f cm^-1; "
      "%.0f %% are more than 5 cm^-1 away" % (np.median(gaps), 100 * np.mean(gaps > 5)))

distance, cm^-1   binned: mean match (range)        kernel: overlap
             0             1.00 (0 to 1)                   1.00
             2             0.90 (0 to 1)                   1.00
             5             0.75 (0 to 1)                   0.98
            10             0.50 (0 to 1)                   0.94
            20             0.00 (0 to 0)                   0.78
            40             0.00 (0 to 0)                   0.37

B97-3c query lines: median distance to the nearest library line 3.7 cm^-1; 40 % are more than 5 cm^-1 away


Two lines 5 cm^-1 apart share a bin for three positions of the edge in four, and have nothing in
common for the fourth. The kernel keeps 98 % of their overlap wherever the lines sit. The lines of
the B97-3c queries are a median 3.7 cm^-1 from the nearest library line, and 40 % of them are more
than 5 cm^-1 away. A histogram therefore loses part of the matching lines to the bin edges. Wider
bins lose fewer of them, but they also merge the nearby lines that tell one acid from another.

A peak table has the same all-or-nothing match at its tolerance. It also drops the weaker lines
and ignores the intensities. The kernel uses every line with its intensity and gives a near line
a partial match. That is what the +12.7 and +22.5 points measure.

## The limit is the query, not the search

With a B97-3c query, the kernel almost always finds the acid. With a GFN2-xTB query it finds it in
fewer than half of the cases. The search is the same, so the difference lies in how far the query
is from the library.

A real measured spectrum is further still. A computed spectrum belongs to one molecule in the gas
phase. A measured spectrum of a carboxylic acid comes from hydrogen-bonded pairs in a solid or a
solution and has a baseline. Two earlier benchmarks measured this on the open Chemotion
collection of measured infrared spectra (`spectra_measured`, `spectra_computed`). The cell reads
their stored results.

In [6]:
with open("benchmarks/results/spectra_measured/measured.json", encoding="utf-8") as fh:
    m = json.load(fh)
with open("benchmarks/results/spectra_computed/subset.json", encoding="utf-8") as fh:
    c = json.load(fh)
n = m["nested"]["all forms"]
print("measured query, measured library (another sample), %d compounds: first in %.1f %%, "
      "first ten %.1f %%" % (m["compounds"], 100 * n["first"], 100 * n["top10"]))
for level, r in c["results"].items():
    print("measured query, %-21s library, %d compounds: first in %.1f %%, first ten %.1f %%, median rank %g"
          % (level, r["library"], 100 * r["nested"]["first"], 100 * r["nested"]["top10"],
             r["median_rank"]))
print("chance: %.1f %%" % (100 * c["chance"]))

measured query, measured library (another sample), 1912 compounds: first in 87.6 %, first ten 95.4 %
measured query, GFN2-xTB acc=1 ohess  library, 230 compounds: first in 11.3 %, first ten 37.2 %, median rank 20
measured query, B97-3c TightSCF       library, 230 compounds: first in 32.3 %, first ten 71.9 %, median rank 3
chance: 0.4 %


Against a library of measured spectra (another sample of the same compound), a measured query
finds its compound first in 87.6 % of the cases, among 1,912 compounds. Against a computed
library of 230 small compounds the share falls to 32.3 % with B97-3c and to 11.3 % with GFN2-xTB;
chance is 0.4 %. With B97-3c the right compound is among the first ten in 71.9 %, and its median
rank is 3.

So on a real measurement the kernel gives a short list, not an answer. The level of the computed
library is a large part of the limit (32.3 against 11.3 %). The rest is the distance between one
molecule in the gas phase and a real sample, which no search can repair. These two benchmarks
compare the slopes of the curves rather than the curves, because a measured curve has a
baseline; the manual gives the details.

## What moves the spectrum: pairing the structure with the spectrum

The last question uses no target at all. A **kernel canonical correlation analysis** (kernel CCA)
takes two descriptions of the same acids: here a structural kernel (the five fields under an
envelope of 10 A around the carbonyl carbon) and the spectrum kernel. It finds pairs of
directions, one in each description, whose values over the acids correlate as strongly as
possible. Each pair is then read by asking what property of the acid it follows.

On the acids it was fitted on, the correlation of a pair is close to 1 whatever the descriptions
hold. So the benchmark scores each pair on held-out substituents and compares it with 100 runs in
which the pairing of the two descriptions was shuffled. That run takes several minutes; the cell
reads its stored result.

A pair that follows a property is only news if neither description shows that property on its
own. So the cell also computes three references live: the carbonyl band itself, the principal
components of the spectra alone (scored on held-out acids), and the components of the structural
kernel alone (a kernel principal component analysis).

In [7]:
with open("benchmarks/results/spectra_bridge/bridge.json", encoding="utf-8") as fh:
    a = json.load(fh)["a"]["views"]["five"]
print("kernel CCA, held-out correlation of each pair (shuffled pairing, 95 % level):")
for i, (rho, q95) in enumerate(zip(a["held_out_rho"], a["null_q95"])):
    print("  pair %d: %+.3f  (%+.3f)" % (i + 1, rho, q95))

mols, names_a, fields5, targets, _fs, table_a = BR.B.read(BR.B.DATA, structures=BR.B.STRUCTURES)
spectra = BR.IR.read_spectra(BR.LIBRARY, names_a)
spectra = [spectra[int(i)] for i in targets.molecule]
width, power = BR.KCCA_SPECTRUM
grid = BR.SP.Grid.for_width(BR.RANGE[0], BR.RANGE[1], width)
Z = BR.unit_rows(BR.SP.feature_matrix(spectra, grid, width, power))
folds = BR.substituent_folds(table_a, BR.KCCA_FOLDS, 0)
spectrum_pcs = BR.held_out_components(Z, folds)
structure_pcs = BR.kernel_components(BR.IR.structure_kernels(fields5, targets)[a["width"]])
props = {"the Hammett constant": np.asarray(table_a["sigma"], float),
         "para (1) or meta (0)": np.array([p == "para" for p in table_a["position"]], float)}
band = np.asarray(targets.y, float)


def strongest(M, v):
    r = [np.corrcoef(M[:, j], v)[0, 1] for j in range(M.shape[1])]
    j = int(np.argmax(np.abs(r)))
    return "%+.2f (component %d)" % (r[j], j + 1)


print("\n%-22s %10s %22s %22s %9s %9s" % ("correlation with", "band", "spectra alone",
                                          "structure alone", "CCA 1", "CCA 2"))
for name, v in props.items():
    cca = a["reading"][name]["spectrum"]
    print("%-22s %+10.2f %22s %22s %+9.2f %+9.2f" % (name, np.corrcoef(band, v)[0, 1],
                                                    strongest(spectrum_pcs, v),
                                                    strongest(structure_pcs, v), cca[0], cca[1]))
print("\n(%.0f s in all)" % (time.time() - start))

kernel CCA, held-out correlation of each pair (shuffled pairing, 95 % level):
  pair 1: +0.822  (+0.190)
  pair 2: +0.804  (+0.202)
  pair 3: +0.700  (+0.212)
  pair 4: +0.521  (+0.183)



correlation with             band          spectra alone        structure alone     CCA 1     CCA 2
the Hammett constant        +0.87    +0.75 (component 1)    +0.47 (component 4)     -0.83     -0.18
para (1) or meta (0)        -0.29    +0.45 (component 5)    -0.60 (component 2)     +0.03     +0.84

(44 s in all)


All four pairs are far above the shuffled pairing. Pair 1 follows the Hammett constant (-0.83 on
the spectral side; the sign of a pair is free). Pair 2 follows the position of the substituent,
para against meta (+0.84). The benchmark reads pair 2 in the region below 1550 cm^-1, where the
bending modes of the ring hydrogens lie, the region a chemist reads for the substitution pattern
of a benzene ring.

The references split this result in two.

- **The Hammett pair is already in the spectrum.** The carbonyl band itself follows the Hammett
  constant at +0.87, which is what plan item SP2 found. The first principal component of the
  spectra alone follows it at +0.75, with no structure and no pairing. Pair 1 is mostly the main
  direction along which these spectra vary.
- **The position pair is new.** No component of the spectra alone follows para against meta
  beyond 0.45, and no component of the structure alone beyond 0.60. Only the pairing finds a
  direction that follows it at +0.84. This is what a kernel CCA adds: it finds what the two
  descriptions share, even when it is not the largest variation in either of them.

## What to take from this notebook

- A library search of spectra has everyday references: binned spectra and a peak-table search.
  Measure them before claiming more. Here the spectrum kernel beats both in every split: 99.6 %
  against 86.8 % with a B97-3c query, and 42.8 % against 32.9 % with a GFN2-xTB query.
- The kernel wins because its match falls smoothly with the distance between two lines. A bin
  edge or a tolerance gives all or nothing.
- Give a reference the same chance as the model. When it chooses the edge of its grid, widen the
  grid. Here that raised the binned search from 23.5 to 32.9 % and halved the kernel's lead.
- The query decides more than the search: 99.6 % with B97-3c, 42.8 % with GFN2-xTB, and 32.3 %
  for a measured spectrum against a B97-3c library of 230 compounds.
- Without a target, a kernel CCA of structure and spectrum found the Hammett constant and the
  substitution position. The first is already the main direction of the spectra alone. The
  second is found only by pairing the two.

The references of this notebook are a part of the benchmark of plan item SP3; this command
measures them again and writes them to `benchmarks/results/spectra_bridge/table.md` (about one
minute):

```
python benchmarks/run_spectra_bridge.py --part reference
```

The command line does not offer a spectrum search. The kernel is `feature_matrix` in
`cmfqsar.spectrum`; the searches and the choice on calibration acids are `reference_settings` and
`choose_on_calibration` in `benchmarks/run_spectra_bridge.py`.

Where to read further in the manual: *The spectrum as the bridge: which acid fits a spectrum, and
what moves a band* has both parts of plan item SP3 and these references; *A measured spectrum
against a library of computed spectra* has the measured queries; *A spectrum as a
one-dimensional field: the kernel and the co-spectrum* has the kernel itself.